# Đánh giá 4 checkpoint VCM trên SFU-HW Class D

Notebook này **không train lại**. Mỗi checkpoint λ=2/4/8/16 được chạy ở nhiều QP, sau đó gộp các điểm thật, lọc Pareto, vẽ PCHIP và tính BD-rate so với DCVC-RT gốc và HEVC/x265.

Chọn Colab `Runtime > Change runtime type > T4 GPU`. Chuẩn bị thư mục Google Drive:

```text
MyDrive/model_moi/SVC/vcm_eval/
├── frames/
├── labels/
├── manifest.json
├── checkpoints/2.pth.tar
├── checkpoints/4.pth.tar
├── checkpoints/8.pth.tar
├── checkpoints/16.pth.tar
├── checkpoints/SVC/checkpoint_base_1.pth.tar
├── checkpoints/SVC/checkpoint_base_2.pth.tar
├── checkpoints/SVC/checkpoint_base_3.pth.tar
├── checkpoints/SVC/checkpoint_base_4.pth.tar
├── weights/cvpr2025_image.pth.tar
├── weights/cvpr2025_video.pth.tar
```

Mọi phương pháp phải dùng đúng cùng manifest, frame, nhãn và cấu hình YOLO. Các marker trên hình là điểm đo thật; PCHIP chỉ làm đường dẫn cong.


In [ ]:
from google.colab import drive
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Hãy bật T4 GPU trong Runtime > Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/model_moi/SVC/vcm_eval')
QPS_PROPOSED = [0, 21, 42, 63]   # 4 checkpoint x 4 QP = 16 điểm thật
QPS_ORIGINAL = [0, 10, 21, 32, 42, 53, 63]
QPS_HEVC = [22, 27, 32, 37, 42, 47]
RUN_PROPOSED = True
RUN_ORIGINAL = True
RUN_HEVC = True
RUN_SVC = True
RAW_MAP5095_PERCENT = None  # ví dụ 42.26 nếu muốn thêm đường raw YOLO

assert DRIVE_ROOT.is_dir(), f'Không thấy {DRIVE_ROOT}'


In [ ]:
# Cài project và entropy coder. CUDA inference extension là tùy chọn.
import os, shutil, subprocess, sys

PROJECT = Path('/content/svc_eval')
YOLO_REPO = Path('/content/yolov5')
REPO = 'https://github.com/uetot1/bla.git'
YOLO_REPO_URL = 'https://github.com/ultralytics/yolov5.git'
COMMIT = '1e5b748e865c60622685a8b0c3b331936a3f0580'

if not (PROJECT / '.git').is_dir():
    subprocess.run(['git', 'clone', REPO, str(PROJECT)], check=True)
subprocess.run(['git', 'fetch', '--all'], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT], cwd=PROJECT, check=True)
if not (YOLO_REPO / '.git').is_dir():
    subprocess.run(
        ['git', 'clone', '--branch', 'v7.0', '--depth', '1', YOLO_REPO_URL, str(YOLO_REPO)],
        check=True,
    )
else:
    subprocess.run(['git', 'fetch', '--tags'], cwd=YOLO_REPO, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'v7.0'], cwd=YOLO_REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'setuptools<81'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(YOLO_REPO / 'requirements.txt')], check=True)

# setup.py chỉ dùng cú pháp Python phổ thông; nới metadata cho runtime Colab Python 3.10/3.11.
setup_py = PROJECT / 'dcvc_rt/src/cpp/setup.py'
text = setup_py.read_text(encoding='utf-8').replace('python_requires=">=3.12"', 'python_requires=">=3.10"')
setup_py.write_text(text, encoding='utf-8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(setup_py.parent)], check=True)

if shutil.which('x265') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'x265', 'ffmpeg'], check=True)

os.chdir(PROJECT)
print('Project:', PROJECT)
print('YOLOv5 v7.0:', YOLO_REPO)
print('Python:', sys.version)
print('x265:', shutil.which('x265'))


In [ ]:
# Dataset đã giải nén sẵn trong Google Drive; đọc trực tiếp thư mục này.
import json

DATA_DIR = DRIVE_ROOT

MANIFEST = DATA_DIR / 'manifest.json'
IMAGE_CKPT = DRIVE_ROOT / 'weights/cvpr2025_image.pth.tar'
ORIGINAL_CKPT = DRIVE_ROOT / 'weights/cvpr2025_video.pth.tar'
SVC_CKPT_DIR = DRIVE_ROOT / 'checkpoints/SVC'
SVC_CHECKPOINTS = [SVC_CKPT_DIR / f'checkpoint_base_{index}.pth.tar' for index in range(1, 5)]
YOLO_WEIGHTS = PROJECT / 'yolov5s.pt'
RESULTS = DRIVE_ROOT / 'results_dense'
RESULTS.mkdir(parents=True, exist_ok=True)

required = [MANIFEST, IMAGE_CKPT, ORIGINAL_CKPT, YOLO_WEIGHTS]
required += [DRIVE_ROOT / f'checkpoints/{lam}.pth.tar' for lam in (2, 4, 8, 16)]
if RUN_SVC:
    required += SVC_CHECKPOINTS
missing = [str(path) for path in required if not path.is_file()]
assert not missing, 'Thiếu file:\n' + '\n'.join(missing)
manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
print('Dataset:', DATA_DIR)
print('Manifest keys:', sorted(manifest))
print('Results:', RESULTS)


In [ ]:
# Chuyển schema checkpoint notebook (p_net/student_front) sang schema evaluator.
CONVERTED = Path('/content/converted_checkpoints')
CONVERTED.mkdir(parents=True, exist_ok=True)
converted_paths = {}

for lam in (2, 4, 8, 16):
    source = DRIVE_ROOT / f'checkpoints/{lam}.pth.tar'
    target = CONVERTED / f'lambda_{lam}.pth.tar'
    checkpoint = torch.load(source, map_location='cpu', weights_only=False)
    assert {'p_net', 'student_front'} <= checkpoint.keys(), f'Sai schema: {source}'
    converted = {
        'schema_version': 7,
        'state_dict': checkpoint['p_net'],
        'cloned_frontend_state_dict': checkpoint['student_front'],
        'hierarchical_qp': True,
        'lambda_task': float(checkpoint.get('lambda_machine', lam)),
        'trainable_components': ['dcvc_rt_dmc'],
        'feature_objective': {
            'task_model': 'yolov5s',
            'cloned_frontend_last_layer': 4,
            'layer_indices': [4],
        },
    }
    torch.save(converted, target)
    converted_paths[lam] = target
    print(f'λ={lam}: epoch={int(checkpoint.get("epoch", -1)) + 1}, {target}')


In [ ]:
# Preflight và hàm chạy; JSON hoàn chỉnh cùng QP sẽ được bỏ qua khi rerun.
import re, shlex
from collections import deque
import MLCodec_extensions_cpp
from dcvc_rt.src.utils.vcm_eval_dataset import AnnotatedVideoDataset

evaluation_sequences = list(AnnotatedVideoDataset(DATA_DIR, MANIFEST))
sequence_frames = {sequence.name: sequence.frame_count for sequence in evaluation_sequences}
print('Evaluation sequences:', sequence_frames)
assert (YOLO_REPO / 'hubconf.py').is_file(), f'Thiếu YOLO hubconf.py: {YOLO_REPO}'
assert evaluation_sequences, 'Manifest không có sequence nào'
assert min(sequence_frames.values()) >= 65, f'Class D thiếu frame: {sequence_frames}'
print('DMCI size (MB):', round(IMAGE_CKPT.stat().st_size / 2**20, 1))
print('DMC original size (MB):', round(ORIGINAL_CKPT.stat().st_size / 2**20, 1))

def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', value).strip('_') or 'method'

def result_complete(path, qps):
    if not path.is_file():
        return False
    data = json.loads(path.read_text(encoding='utf-8'))
    saved = data.get('codec_config', {}).get('base_qps', data.get('codec_config', {}).get('qps'))
    return list(saved or []) == list(qps) and len(data.get('points', [])) == len(qps)

def run_and_stream(command):
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(YOLO_REPO) + os.pathsep + environment.get('PYTHONPATH', '')
    process = subprocess.Popen(
        command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=environment,
    )
    tail = deque(maxlen=60)
    for line in process.stdout:
        print(line, end='')
        tail.append(line.rstrip())
    return_code = process.wait()
    if return_code:
        raise RuntimeError(
            f'Command failed with exit code {return_code}:\n' + '\n'.join(tail)
        )

def run_codec(method, video_checkpoint, qps, output_subdir):
    output_dir = RESULTS / output_subdir
    result_path = output_dir / f'{safe_name(method)}_results.json'
    if result_complete(result_path, qps):
        print('SKIP, đã hoàn chỉnh:', result_path)
        return result_path
    command = [
        sys.executable, 'evaluate_vcm.py', '--mode', 'codec',
        '--data-dir', str(DATA_DIR), '--dataset-manifest', str(MANIFEST),
        '--image-ckpt', str(IMAGE_CKPT), '--video-ckpt', str(video_checkpoint),
        '--qps', *map(str, qps), '--reset-interval', '32',
        '--minimum-sequence-frames', '65', '--force-zero-thres', '0.12',
        '--codec-precision', 'fp16',
        '--yolov5-weights', str(YOLO_WEIGHTS), '--detector-size', '640',
        '--confidence-threshold', '0.001', '--nms-iou-threshold', '0.6',
        '--max-detections', '300', '--method-name', method,
        '--output-dir', str(output_dir), '--bitstream-dir', '/content/vcm_bitstreams',
    ]
    print('RUN:', shlex.join(command))
    run_and_stream(command)
    assert result_complete(result_path, qps), f'Kết quả chưa hoàn chỉnh: {result_path}'
    return result_path


## Chạy Proposed

Cell này chạy 16 operating points thật. Nếu muốn dày hơn và runtime cho phép, đổi `QPS_PROPOSED` thành `[0, 16, 32, 48, 63]`; helper sẽ nhận ra danh sách QP thay đổi và chạy lại Proposed.


In [ ]:
proposed_results = []
if RUN_PROPOSED:
    for lam in (2, 4, 8, 16):
        proposed_results.append(
            run_codec(f'lambda_{lam}', converted_paths[lam], QPS_PROPOSED, f'lambda_{lam}')
        )
else:
    proposed_results = [RESULTS / f'lambda_{lam}/lambda_{lam}_results.json' for lam in (2, 4, 8, 16)]


## Chạy DCVC-RT gốc


In [ ]:
original_result = RESULTS / 'original/Original_DCVC-RT_results.json'
if RUN_ORIGINAL:
    original_result = run_codec('Original DCVC-RT', ORIGINAL_CKPT, QPS_ORIGINAL, 'original')


## Chạy HEVC/x265

x265 có progress checkpoint theo từng sequence/QP nên có thể tiếp tục sau khi Colab ngắt.


In [ ]:
hevc_dir = RESULTS / 'hevc'
hevc_result = hevc_dir / 'HEVC_x265_results.json'
if RUN_HEVC and not result_complete(hevc_result, QPS_HEVC):
    hevc_dir.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, 'evaluate_hevc.py', '--data-dir', str(DATA_DIR),
        '--dataset-manifest', str(MANIFEST), '--x265-encoder', 'x265',
        '--ffmpeg', 'ffmpeg', '--qps', *map(str, QPS_HEVC),
        '--bit-depth', '8', '--chroma-format', '420', '--preset', 'medium',
        '--x265-extra-arg=--ref=4', '--x265-extra-arg=--keyint=32',
        '--x265-extra-arg=--min-keyint=32',
        '--yolov5-weights', str(YOLO_WEIGHTS), '--detector-size', '640',
        '--detector-batch-size', '16', '--confidence-threshold', '0.001',
        '--nms-iou-threshold', '0.6', '--max-detections', '300',
        '--method-name', 'HEVC_x265', '--output-dir', str(hevc_dir),
        '--bitstream-dir', '/content/hevc_bitstreams',
        '--encoder-log-dir', str(hevc_dir / 'logs'), '--work-dir', '/content/hevc_work',
        '--progress-checkpoint', str(hevc_dir / 'HEVC_x265_progress.pt'),
        '--resume', '--keep-progress-checkpoint',
    ]
    print('RUN:', shlex.join(command))
    run_and_stream(command)
assert result_complete(hevc_result, QPS_HEVC), f'HEVC chưa hoàn chỉnh: {hevc_result}'


## Chạy SVC gốc (Base layer)

Bốn checkpoint chính thức tạo bốn điểm rate. Repo SVC phát hành chỉ tính entropy-likelihood BPP, vì vậy đường này được ghi rõ **Estimated BPP**, không giả thành Actual BPP.


In [ ]:
svc_dir = RESULTS / 'svc_original'
svc_result = svc_dir / 'Original_SVC_Base_estimated_results.json'
svc_complete = svc_result.is_file() and len(json.loads(svc_result.read_text(encoding='utf-8')).get('points', [])) == 4
if RUN_SVC and not svc_complete:
    command = [
        sys.executable, 'evaluate_svc_legacy.py',
        '--data-dir', str(DATA_DIR), '--dataset-manifest', str(MANIFEST),
        '--checkpoints', *map(str, SVC_CHECKPOINTS),
        '--yolov5-weights', str(YOLO_WEIGHTS), '--gop', '32',
        '--detector-size', '640', '--confidence-threshold', '0.001',
        '--nms-iou-threshold', '0.6', '--max-detections', '300',
        '--output-dir', str(svc_dir),
    ]
    print('RUN:', shlex.join(command))
    run_and_stream(command)
assert svc_result.is_file(), f'SVC chưa hoàn chỉnh: {svc_result}'


## Gộp Pareto, vẽ PCHIP và tính BD-rate

BD-rate âm nghĩa là candidate tiết kiệm bitrate tại cùng mAP. Chỉ miền mAP giao nhau được tích phân. Các hàng liên quan SVC được đánh dấu exploratory vì SVC phát hành Estimated BPP, còn ba phương pháp kia dùng Actual BPP.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from IPython.display import display
from dcvc_rt.src.utils.bd_rate import compute_bd_rate, pareto_front

def load_result(path):
    data = json.loads(Path(path).read_text(encoding='utf-8'))
    assert data.get('schema_version') in (6, 7)
    assert len(data.get('points', [])) >= 4
    return data

parts = [load_result(path) for path in proposed_results]
original = load_result(original_result)
hevc = load_result(hevc_result)
svc = load_result(svc_result)

compatibility_keys = ('evaluation_id', 'task_model', 'protocol', 'ground_truth', 'detector_config', 'comparison_scope')
for data in [*parts, hevc]:
    for key in compatibility_keys:
        assert json.dumps(data.get(key), sort_keys=True) == json.dumps(original.get(key), sort_keys=True), f'Protocol lệch ở {key}: {data["method"]}'
    assert {int(p['coded_frames']) for p in data['points']} == {int(p['coded_frames']) for p in original['points']}, f'Số frame lệch: {data["method"]}'
assert svc['evaluation_id'] == original['evaluation_id'], 'SVC dùng khác dataset/manifest'
assert {int(p['coded_frames']) for p in svc['points']} == {int(p['coded_frames']) for p in original['points']}, 'SVC dùng khác số frame'

proposed = {
    'method': 'Proposed DCVC-RT-VCM',
    'points': [
        {**point, 'source_method': part['method']}
        for part in parts for point in part['points']
    ],
}
curves = [proposed, original, hevc, svc]
rate_keys = {data['method']: ('estimated_bpp' if data is svc else 'actual_bpp') for data in curves}
rate_sources = {data['method']: ('estimated' if data is svc else 'actual') for data in curves}
metric = 'map5095'
pareto = {}
for data in curves:
    rates = [float(point[rate_keys[data['method']]]) for point in data['points']]
    quality = [float(point[metric]) for point in data['points']]
    pareto[data['method']] = pareto_front(rates, quality)

comparison_dir = RESULTS / 'comparison_dense'
comparison_dir.mkdir(parents=True, exist_ok=True)
figure, axis = plt.subplots(figsize=(10, 6.5))
markers = ('o', 's', '^', 'D')
line_styles = ('-', '-', '-', '--')
for index, data in enumerate(curves):
    rates, quality = pareto[data['method']]
    quality_grid = np.linspace(quality[0], quality[-1], 250)
    rate_grid = 10 ** PchipInterpolator(quality, np.log10(rates))(quality_grid)
    line = axis.plot(rate_grid, quality_grid * 100, linewidth=2.3, linestyle=line_styles[index], label=data['method'])[0]
    axis.scatter(rates, quality * 100, marker=markers[index], s=55, color=line.get_color(), zorder=3)
if RAW_MAP5095_PERCENT is not None:
    axis.axhline(float(RAW_MAP5095_PERCENT), linestyle='--', color='tab:blue', label=f'Raw YOLOv5s ({RAW_MAP5095_PERCENT:.2f}%)')
axis.set(xlabel='BPP (Actual: DCVC/x265; Estimated: Original SVC)', ylabel='mAP@[0.5:0.95] (%)', title='SFU-HW Class D — Object Detection mAP vs BPP')
axis.grid(True, alpha=0.3)
axis.legend()
figure.tight_layout()
plot_path = comparison_dir / 'rd_curve_bpp_map5095_with_svc.png'
figure.savefig(plot_path, dpi=200)
plt.show()

bd_rows = []
def add_bd_row(anchor, candidate):
    anchor_rate, anchor_quality = pareto[anchor['method']]
    candidate_rate, candidate_quality = pareto[candidate['method']]
    lower = max(float(anchor_quality.min()), float(candidate_quality.min()))
    upper = min(float(anchor_quality.max()), float(candidate_quality.max()))
    anchor_source = rate_sources[anchor['method']]
    candidate_source = rate_sources[candidate['method']]
    bd_rows.append({
        'anchor': anchor['method'],
        'candidate': candidate['method'],
        'metric': metric,
        'anchor_rate_source': anchor_source,
        'candidate_rate_source': candidate_source,
        'validity': 'comparable_actual_bpp' if anchor_source == candidate_source == 'actual' else 'exploratory_mixed_actual_estimated',
        'overlap_mAP_min_percent': lower * 100,
        'overlap_mAP_max_percent': upper * 100,
        'bd_rate_percent': compute_bd_rate(anchor_rate, anchor_quality, candidate_rate, candidate_quality) if lower < upper else None,
        'interpretation': 'negative means bitrate saving',
    })
for anchor in (original, hevc):
    add_bd_row(anchor, proposed)
for anchor in (original, hevc, proposed):
    add_bd_row(anchor, svc)
bd_frame = pd.DataFrame(bd_rows)
display(bd_frame)
bd_frame.to_csv(comparison_dir / 'bd_rate_map5095.csv', index=False)
(comparison_dir / 'bd_rate_map5095.json').write_text(json.dumps(bd_rows, indent=2), encoding='utf-8')

rd_rows = []
for data in curves:
    for point in data['points']:
        rd_rows.append({
            'method': data['method'], 'source_method': point.get('source_method', data['method']),
            'base_qp': point.get('base_qp'), 'checkpoint_number': point.get('checkpoint_number'),
            'rate_bpp': point[rate_keys[data['method']]], 'rate_source': rate_sources[data['method']],
            'actual_bpp': point.get('actual_bpp'), 'estimated_bpp': point.get('estimated_bpp'),
            'map50': point['map50'], 'map5095': point['map5095'],
        })
pd.DataFrame(rd_rows).to_csv(comparison_dir / 'rd_points.csv', index=False)
print('Đã lưu:', comparison_dir)


In [ ]:
# Tải gói kết quả về máy.
from google.colab import files
archive = shutil.make_archive('/content/class_d_dense_evaluation', 'zip', root_dir=RESULTS)
print(archive)
files.download(archive)
